
## 01MIAR - Actividad Video Valencia Pollution - Code

*Ivan Fuertes*

## Web API pública
- Interfaz de programación que se pone a disposición de cualquier desarrollador para acceder a datos o funcionalidades
### Valencia Open Data
- https://opendata.vlci.valencia.es/
#### Estaciones contaminación atmosférica
- https://opendata.vlci.valencia.es/dataset/estacions-contaminacio-atmosferiques-estaciones-contaminacion-atmosfericas

#### API Creada con CKAN
- https://docs.ckan.org/en/2.11/

### Requests
- Librería para hacer peticiones Http a web
- https://requests.readthedocs.io/en/latest/

In [ ]:
import requests

In [ ]:
# function to get a json response from a url + params

def make_request(base_url, params):
    response = requests.get(base_url, params = params)
    if response:
        return response.json()
    else:
        print(response)
        raise Exception("Error Downloading JSON")

- URL Base del servidor
- https://geoportal.valencia.es/server/rest/services/OPENDATA/

In [ ]:
server_url = "https://geoportal.valencia.es/server/rest/services/OPENDATA/"

- Prueba de la llamada a la API
- https://geoportal.valencia.es/server/rest/services/OPENDATA/MedioAmbiente/MapServer/156/query?where=1=1&outFields=*&f=json

In [ ]:
base_url = server_url + "MedioAmbiente/MapServer/156/query"

#### Parámetros del Query
- `'f' = 'json'` - formato del fichero de respuesta
- `'outFields' = '*'` - campos en la respuesta, todos
- `'where' = '1=1'` - filtro

In [ ]:
params = {'f' : 'json', 'outFields' : '*', 'where' : '1=1'}
print(base_url)

#### Realizar petición y explorar resultados
- `'fieldAliases'` = nombres de campos y sus alias
- `'fields'` = nombres de campos, alias, tipos y longitudes
- `'features'` = lista de estaciones y lecturas
  - `'attributes'` = lectura de una estación
  - `'geometry'` = coordenadas geométricas de una estación (sistema proyectado UTM)

In [ ]:
import json   # to acquire and format json string

response = make_request(base_url, params)

print(json.dumps(response, indent = 2))

In [ ]:
from pyproj import Transformer   # to converto from UTM to GPS lat+long
import pandas as pd

transformer = Transformer.from_crs("EPSG:32630", "EPSG:4326", always_xy=True)

df = pd.DataFrame()

for feature in response['features']:
    serie = pd.Series(feature['attributes'])

    # transform from UTM to GPS
    serie["lon"], serie["lat"] = transformer.transform(feature['geometry']['x'], feature['geometry']['y'])
    
    df = pd.concat([df, serie.to_frame().T], axis = 0, ignore_index = True)

# format datetime from epoch (as miliseconds) to timestamp (GMT)
df['fecha_carg'] = pd.to_datetime(df['fecha_carg'], unit = 'ms')

In [ ]:
display(df.head(3))

### Tomar muestras de los datos cada hora durante un día completo

In [ ]:
sleep_time = 60 * 60 # 60 minutes in seconds
total_time = (24 * 60 * 60) + sleep_time  # 1 day to take samples (plus one sleep extra just in case)
current_time = 0 # current time before ending sampling

In [ ]:
# endpoints
server_url = "https://geoportal.valencia.es/server/rest/services/OPENDATA/"
base_url = server_url + "MedioAmbiente/MapServer/156/query"

In [ ]:
# params
params = {'f' : 'json', 'outFields' : '*', 'where' : '1=1'}

In [ ]:
# path to save dataset in csv format
import os
from time import sleep

path_csv = ['res', 'valencia_pollution_dataset.csv']
path_csv_solved = os.path.join(*path_csv)
first_time = True # flag for one use only to save headers in csv

In [ ]:
# main loop
while current_time < total_time:
    pollution_list = make_request(base_url, params)['features']

    print(f"Current Time = {current_time}, Records Processed = {len(pollution_list)}")
    
    df = pd.DataFrame()
    for feature in pollution_list:
        serie = pd.Series(feature['attributes'])
        serie["lon"], serie["lat"] = transformer.transform(feature['geometry']['x'], feature['geometry']['y'])
        df = pd.concat([df, serie.to_frame().T], axis = 0, ignore_index = True)
        
    df['fecha_carg'] = pd.to_datetime(df['fecha_carg'], unit = 'ms')

    df.to_csv(path_csv_solved, sep=',', header=first_time, mode='a', index=False)

    first_time = False
    sleep(sleep_time)
    current_time += sleep_time

In [ ]:
display(df.head(5))

In [ ]:
display(df.info())

In [ ]:
display(df.describe(include='all'))